# 03 · Data Warehouse: modelo en estrella (R3 · Hito 2)
**Proyecto semestral · Sala de crisis COVID-19 · Gestión de Datos IN1232C · 2026-2**

Requiere `01_ingesta` y `02_calidad`. Seguimos los mismos pasos del taller de la S10:
1. Diseño con el método de Kimball.
2. ETL: dimensión de fecha, dimensión de territorio y tabla de hechos.
3. Carga en SQLite (`to_sql`) y conciliación con la fuente.
4. Operaciones OLAP: roll-up, drill-down, slice, dice y pivot.

## 1 · Diseño (método de Kimball)
| Paso | Pregunta | Nuestra respuesta |
|---|---|---|
| 1 · Proceso de negocio | ¿Qué actividad medimos? | El reporte diario de casos y fallecidos COVID-19 |
| 2 · Grano | ¿Qué es UNA fila de hechos? | **Un territorio en un día** (país, región/estado o condado) |
| 3 · Dimensiones | ¿Cuándo, dónde? | `dim_fecha` · `dim_territorio` (país → región → condado, con población) |
| 4 · Hechos | ¿Qué se mide y se suma? | Casos y fallecidos acumulados, casos y fallecidos nuevos, banderas de calidad |

```
              dim_fecha                 dim_territorio
           (fecha_key)               (territorio_key)
                    \                 /
                     fact_covid_diario
          fecha_key · territorio_key · acumulados · nuevos · banderas
```

### ¿Por qué SQLite y no una base NoSQL? (S4)
| Criterio | SQLite (elegido) | MongoDB (NoSQL documento) |
|---|---|---|
| Forma de los datos | Tablas con columnas fijas: lo que tenemos tras unificar los 5 esquemas | Documentos JSON con campos distintos por registro |
| Uso | Leer y resumir millones de filas con JOIN y GROUP BY | Muchas escrituras de documentos sueltos |
| Modelo en estrella | Natural: llaves y JOIN | No tiene JOIN real (`$lookup` limitado) |
| Instalación | Viene con Python (`import sqlite3`), un solo archivo | Requiere un servidor |

Como dice la S4: *antes de elegir NoSQL, descartar que una base relacional resuelva el problema*. Aquí la resuelve. El esquema cambió solo 5 veces y ya está unificado, así que la flexibilidad de MongoDB no aporta.

## Setup

In [1]:
import os
import shutil
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', None)
pd.set_option('display.precision', 2)

from google.colab import drive
drive.mount('/content/drive')

PROYECTO = '/content/drive/MyDrive/proyecto-gestion-datos-2026'
LIMPIOS  = os.path.join(PROYECTO, 'data', 'clean')
DW_DRIVE = os.path.join(PROYECTO, 'data', 'dw')
os.makedirs(DW_DRIVE, exist_ok=True)

# El DW se arma en el disco de la sesión (más rápido) y al final se copia a Drive
DW_LOCAL = '/content/covid_dw.db'

## 2 · Extracción

In [2]:
covid = pd.read_parquet(os.path.join(LIMPIOS, 'covid_diario.parquet'))
uid   = pd.read_parquet(os.path.join(LIMPIOS, 'uid.parquet'))
homol = pd.read_parquet(os.path.join(LIMPIOS, 'homologacion_paises.parquet'))
print('Reportes:', covid.shape, '| UID:', uid.shape, '| Homologación:', homol.shape)

Reportes: (4287473, 17) | UID: (4321, 12) | Homologación: (249, 3)


In [3]:
# Mismas correcciones que en 02_calidad
covid = covid.merge(homol, on='country_region', how='left')
es_ultramar = covid['es_ultramar'] & covid['province_state'].isna()
covid.loc[es_ultramar, 'province_state'] = covid.loc[es_ultramar, 'country_region']
covid.loc[covid['province_state'] == covid['pais'], 'province_state'] = np.nan

# Unicidad: si un territorio aparece dos veces el mismo día, dejamos la última fila del archivo
antes = len(covid)
covid = covid.sort_values(['fecha_reporte', 'fila_origen'])
covid = covid.drop_duplicates(subset=['fecha_reporte', 'pais', 'province_state', 'admin2'], keep='last')
print('Duplicados eliminados:', antes - len(covid))

Duplicados eliminados: 78


## 3 · Dimensión de fecha
Como en el taller: no viene de ningún sistema, se genera con una fila por día. `fecha_key` es la fecha escrita como número AAAAMMDD.

In [4]:
f = pd.date_range('2020-01-22', '2023-03-09')
meses = ['enero', 'febrero', 'marzo', 'abril', 'mayo', 'junio', 'julio', 'agosto',
         'septiembre', 'octubre', 'noviembre', 'diciembre']

dim_fecha = pd.DataFrame({
    'fecha_key':     f.strftime('%Y%m%d').astype(int),
    'fecha':         f.strftime('%Y-%m-%d'),
    'anio':          f.year,
    'trimestre':     f.quarter,
    'mes':           f.month,
    'nombre_mes':    [meses[m - 1] for m in f.month],
    'semana':        f.isocalendar().week.values,
    'dia_semana':    f.dayofweek,                       # 0 = lunes ... 6 = domingo
    'es_fin_semana': (f.dayofweek >= 5).astype(int),
})
print('Días:', len(dim_fecha))
dim_fecha.head()

Días: 1143


,fecha_key,fecha,anio,trimestre,mes,nombre_mes,semana,dia_semana,es_fin_semana
0,20200122,2020-01-22,2020,1,1,enero,4,2,0
1,20200123,2020-01-23,2020,1,1,enero,4,3,0
2,20200124,2020-01-24,2020,1,1,enero,4,4,0
3,20200125,2020-01-25,2020,1,1,enero,4,5,1
4,20200126,2020-01-26,2020,1,1,enero,4,6,1


## 4 · Dimensión de territorio
Parte de la **tabla UID** (4.321 territorios con población, coordenadas e ISO3). La jerarquía es **país → región o estado → condado**. Los territorios de los reportes que no están en la tabla UID se agregan sin población.

In [5]:
dim_territorio = uid[['uid', 'country_region', 'province_state', 'admin2', 'iso3', 'lat', 'long', 'population']].copy()
dim_territorio = dim_territorio.rename(columns={'country_region': 'pais', 'province_state': 'region',
                                                'admin2': 'condado', 'population': 'poblacion'})
dim_territorio['en_tabla_uid'] = 1

# ¿Qué territorios de los reportes no están en la tabla UID?
territorios = covid[['pais', 'province_state', 'admin2']].drop_duplicates()
territorios = territorios.rename(columns={'province_state': 'region', 'admin2': 'condado'})
faltan = territorios.merge(dim_territorio[['pais', 'region', 'condado']], how='left', indicator=True)
faltan = faltan[faltan['_merge'] == 'left_only'].drop(columns='_merge')
faltan['en_tabla_uid'] = 0
print('Territorios en los reportes:', len(territorios), '| sin correspondencia en la tabla UID:', len(faltan))
faltan.head(10)

Territorios en los reportes: 4289 | sin correspondencia en la tabla UID: 176


,pais,region,condado,en_tabla_uid
28,Taiwan*,Taiwan,None,0
83,US,Chicago,None,0
96,Germany,Bavaria,None,0
105,US,"Chicago, IL",None,0
108,US,"Boston, MA",None,0
109,US,"Los Angeles, CA",None,0
110,US,"Orange, CA",None,0
111,US,"Santa Clara, CA",None,0
112,US,"Seattle, WA",None,0
113,US,"Tempe, AZ",None,0


In [6]:
dim_territorio = pd.concat([dim_territorio, faltan], ignore_index=True)

# Nivel de la jerarquía
reglas = [dim_territorio['condado'].notna(), dim_territorio['region'].notna()]
dim_territorio['nivel'] = np.select(reglas, ['condado', 'region'], default='pais')

# ISO3 del país (para cruzar con fuentes externas) y coordenadas (0, 0), que no son reales
iso_pais = uid.loc[uid['province_state'].isna(), ['country_region', 'iso3']].rename(
    columns={'country_region': 'pais', 'iso3': 'iso3_pais'})
dim_territorio = dim_territorio.merge(iso_pais, on='pais', how='left')
dim_territorio.loc[(dim_territorio['lat'] == 0) & (dim_territorio['long'] == 0), ['lat', 'long']] = np.nan

# Llave sustituta del DW
dim_territorio.insert(0, 'territorio_key', range(1, len(dim_territorio) + 1))

print(dim_territorio['nivel'].value_counts())
dim_territorio[dim_territorio['pais'] == 'Chile']

nivel
condado    3363
region      932
pais        202
Name: count, dtype: int64


,territorio_key,uid,pais,region,condado,iso3,lat,long,poblacion,en_tabla_uid,nivel,iso3_pais
74,75,152.0,Chile,None,None,CHL,-35.68,-71.54,1.91e+07,1,pais,CHL
75,76,15201.0,Chile,Antofagasta,None,CHL,-23.65,-70.40,6.08e+05,1,region,CHL
76,77,15202.0,Chile,Araucania,None,CHL,-38.95,-72.33,9.57e+05,1,region,CHL
77,78,15203.0,Chile,Arica y Parinacota,None,CHL,-18.59,-69.48,2.26e+05,1,region,CHL
78,79,15204.0,Chile,Atacama,None,CHL,-27.57,-70.05,2.89e+05,1,region,CHL
79,80,15205.0,Chile,Aysen,None,CHL,-45.99,-73.77,1.03e+05,1,region,CHL
80,81,15206.0,Chile,Biobio,None,CHL,-37.45,-72.14,1.56e+06,1,region,CHL
81,82,15207.0,Chile,Coquimbo,None,CHL,-29.96,-71.34,7.58e+05,1,region,CHL
82,83,15208.0,Chile,Los Lagos,None,CHL,-41.92,-72.14,8.29e+05,1,region,CHL
83,84,15209.0,Chile,Los Rios,None,CHL,-40.23,-72.33,3.85e+05,1,region,CHL


## 5 · Tabla de hechos
* **Grano:** un territorio en un día.
* **Nuevos** = diferencia con el día anterior **del mismo territorio**. El primer día de un territorio queda vacío: así, cuando Chile pasa de 1 fila nacional a 16 regiones (20-may-2020), no aparece un pico falso.
* **Banderas de calidad** (decisiones de 02_calidad): acumulado que baja y archivo defectuoso del 23-may-2022.
* **Linaje:** archivo y fila de origen.

In [7]:
m = covid.rename(columns={'province_state': 'region', 'admin2': 'condado'})
m = m.merge(dim_territorio[['territorio_key', 'pais', 'region', 'condado']],
            on=['pais', 'region', 'condado'], how='left')
assert m['territorio_key'].notna().all(), 'Hay filas sin territorio'

m['fecha_key'] = m['fecha_reporte'].dt.strftime('%Y%m%d').astype(int)
m = m.sort_values(['territorio_key', 'fecha_key'])

m['casos_nuevos']      = m.groupby('territorio_key')['confirmed'].diff()
m['fallecidos_nuevos'] = m.groupby('territorio_key')['deaths'].diff()
m['acum_baja']         = ((m['casos_nuevos'] < 0) | (m['fallecidos_nuevos'] < 0)).astype(int)
m['archivo_defectuoso'] = (m['archivo_origen'] == '05-23-2022.csv').astype(int)

fact_covid = m[['fecha_key', 'territorio_key', 'confirmed', 'deaths', 'casos_nuevos', 'fallecidos_nuevos',
                'acum_baja', 'archivo_defectuoso', 'archivo_origen', 'fila_origen']].rename(
    columns={'confirmed': 'casos_acum', 'deaths': 'fallecidos_acum'})
del m
print(f'Filas de hechos: {len(fact_covid):,}')
fact_covid.head()

Filas de hechos: 4,287,395


,fecha_key,territorio_key,casos_acum,fallecidos_acum,casos_nuevos,fallecidos_nuevos,acum_baja,archivo_defectuoso,archivo_origen,fila_origen
3456,20200224,1,5.0,0.0,NaN,NaN,0,0,02-24-2020.csv,63
3587,20200225,1,5.0,0.0,0.0,0.0,0,0,02-25-2020.csv,67
3726,20200226,1,5.0,0.0,0.0,0.0,0,0,02-26-2020.csv,75
3868,20200227,1,5.0,0.0,0.0,0.0,0,0,02-27-2020.csv,77
4013,20200228,1,5.0,0.0,0.0,0.0,0,0,02-28-2020.csv,79


## 6 · Carga en SQLite y conciliación

In [8]:
dw = sqlite3.connect(DW_LOCAL)

for nombre, df in {'dim_fecha': dim_fecha,
                   'dim_territorio': dim_territorio,
                   'fact_covid': fact_covid}.items():
    df.to_sql(nombre, dw, if_exists='replace', index=False)
    print(f'{nombre:15s} {len(df):>10,} filas cargadas')

dim_fecha            1,143 filas cargadas
dim_territorio       4,497 filas cargadas


fact_covid       4,287,395 filas cargadas


In [9]:
# Conciliación: ¿lo que quedó en el DW cuadra con la fuente?
casos_fuente = covid.loc[covid['fecha_reporte'] == '2023-03-09', 'confirmed'].sum()
casos_dw = pd.read_sql('SELECT SUM(casos_acum) AS c FROM fact_covid WHERE fecha_key = 20230309', dw)['c'][0]

control = pd.DataFrame({
    'control': ['filas', 'días', 'casos acumulados al 9-mar-2023', 'hechos sin fecha', 'hechos sin territorio'],
    'fuente':  [len(covid), covid['fecha_reporte'].nunique(), casos_fuente, 0, 0],
    'DW': [pd.read_sql('SELECT COUNT(*) AS n FROM fact_covid', dw)['n'][0],
           pd.read_sql('SELECT COUNT(DISTINCT fecha_key) AS n FROM fact_covid', dw)['n'][0],
           casos_dw,
           pd.read_sql('SELECT COUNT(*) AS n FROM fact_covid WHERE fecha_key NOT IN (SELECT fecha_key FROM dim_fecha)', dw)['n'][0],
           pd.read_sql('SELECT COUNT(*) AS n FROM fact_covid WHERE territorio_key NOT IN (SELECT territorio_key FROM dim_territorio)', dw)['n'][0]],
})
control['cuadra'] = np.where(control['fuente'] == control['DW'], 'OK', 'REVISAR')
control

,control,fuente,DW,cuadra
0,filas,4.29e+06,4.29e+06,OK
1,días,1.14e+03,1.14e+03,OK
2,casos acumulados al 9-mar-2023,6.77e+08,6.77e+08,OK
3,hechos sin fecha,0.00e+00,0.00e+00,OK
4,hechos sin territorio,0.00e+00,0.00e+00,OK


## 7 · Consultas OLAP
Igual que en el taller: con SQL sobre la estrella armamos un **cubo** (territorio × mes) y luego usamos pandas para rebanarlo.

In [10]:
cubo = pd.read_sql('''
    SELECT t.pais, t.region, t.nivel, t.poblacion, d.anio, d.mes,
           SUM(f.casos_nuevos)      AS casos_nuevos,
           SUM(f.fallecidos_nuevos) AS fallecidos_nuevos
    FROM fact_covid f
    JOIN dim_fecha d      ON d.fecha_key = f.fecha_key
    JOIN dim_territorio t ON t.territorio_key = f.territorio_key
    GROUP BY t.territorio_key, d.anio, d.mes
''', dw)
print('Cubo:', cubo.shape)
cubo.head()

Cubo: (147002, 8)


,pais,region,nivel,poblacion,anio,mes,casos_nuevos,fallecidos_nuevos
0,Afghanistan,None,pais,3.89e+07,2020,2,0.0,0.0
1,Afghanistan,None,pais,3.89e+07,2020,3,161.0,4.0
2,Afghanistan,None,pais,3.89e+07,2020,4,1661.0,56.0
3,Afghanistan,None,pais,3.89e+07,2020,5,13353.0,194.0
4,Afghanistan,None,pais,3.89e+07,2020,6,16265.0,485.0


In [11]:
# ROLL-UP: de territorio × mes a país × año (casos nuevos en Chile por año)
chile = cubo[cubo['pais'] == 'Chile']
print(chile.groupby('anio')['casos_nuevos'].sum().to_string())

anio
2020    6.05e+05
2021    1.20e+06
2022    3.23e+06
2023    1.55e+05


In [12]:
# DRILL-DOWN + SLICE: Chile por región (solo regiones) · fallecidos por 100.000 habitantes en toda la pandemia
regiones = chile[(chile['nivel'] == 'region') & (chile['region'] != 'Unknown')]
por_region = regiones.groupby('region').agg(fallecidos=('fallecidos_nuevos', 'sum'),
                                             poblacion=('poblacion', 'first'))
por_region['fallecidos_x100k'] = (por_region['fallecidos'] / por_region['poblacion'] * 100000).round(1)
por_region = por_region.sort_values('fallecidos_x100k', ascending=False)
por_region['posicion'] = range(1, len(por_region) + 1)
por_region

,fallecidos,poblacion,fallecidos_x100k,posicion
region,,,,
Magallanes,756.0,1.67e+05,454.0,1
Metropolitana,30521.0,7.11e+06,429.1,2
Arica y Parinacota,893.0,2.26e+05,395.0,3
Los Rios,1405.0,3.85e+05,365.1,4
Valparaiso,6426.0,1.82e+06,353.9,5
Tarapaca,1169.0,3.31e+05,353.6,6
Antofagasta,1944.0,6.08e+05,320.0,7
Maule,3340.0,1.04e+06,319.6,8
Los Lagos,2565.0,8.29e+05,309.5,9


In [13]:
# DICE: Biobío y Metropolitana en 2021 · PIVOT: meses en las columnas
dice = regiones[(regiones['region'].isin(['Biobio', 'Metropolitana'])) & (regiones['anio'] == 2021)]
dice.pivot_table(index='region', columns='mes', values='casos_nuevos', aggfunc='sum')

mes,1,2,3,4,5,6,7,8,9,10,11,12
region,,,,,,,,,,,,
Biobio,13687.0,11907.0,22485.0,19708.0,15397.0,12593.0,5297.0,1797.0,1227.0,2056.0,7506.0,6711.0
Metropolitana,24139.0,21079.0,55497.0,77836.0,78902.0,76573.0,20728.0,10628.0,7647.0,23437.0,26307.0,11474.0


In [14]:
# Chile frente a Sudamérica: casos y fallecidos acumulados por 100.000 al 9-mar-2023
sudamerica = ['Argentina', 'Bolivia', 'Brazil', 'Chile', 'Colombia', 'Ecuador', 'Guyana',
              'Paraguay', 'Peru', 'Suriname', 'Uruguay', 'Venezuela']
ultimo = pd.read_sql('''
    SELECT t.pais, SUM(f.casos_acum) AS casos, SUM(f.fallecidos_acum) AS fallecidos
    FROM fact_covid f JOIN dim_territorio t ON t.territorio_key = f.territorio_key
    WHERE f.fecha_key = 20230309
    GROUP BY t.pais
''', dw)
poblacion_pais = dim_territorio[(dim_territorio['nivel'] == 'pais') & (dim_territorio['en_tabla_uid'] == 1)][['pais', 'poblacion']]
ultimo = ultimo.merge(poblacion_pais, on='pais')
sud = ultimo[ultimo['pais'].isin(sudamerica)].copy()
sud['casos_x100k'] = (sud['casos'] / sud['poblacion'] * 100000).round(0)
sud['fallecidos_x100k'] = (sud['fallecidos'] / sud['poblacion'] * 100000).round(1)
sud.sort_values('fallecidos_x100k', ascending=False)

,pais,casos,fallecidos,poblacion,casos_x100k,fallecidos_x100k
141,Peru,4.49e+06,219539.0,3.30e+07,13610.0,665.8
36,Chile,5.19e+06,64273.0,1.91e+07,27162.0,336.2
24,Brazil,3.71e+07,699276.0,2.13e+08,17443.0,329.0
7,Argentina,1.00e+07,130472.0,4.52e+07,22225.0,288.7
38,Colombia,6.36e+06,142339.0,5.09e+07,12498.0,279.7
140,Paraguay,8.08e+05,19878.0,7.13e+06,11334.0,278.7
171,Suriname,8.25e+04,1404.0,5.87e+05,14058.0,239.3
191,Uruguay,1.03e+06,7617.0,3.47e+06,29775.0,219.3
53,Ecuador,1.06e+06,36014.0,1.76e+07,5992.0,204.1
21,Bolivia,1.19e+06,22365.0,1.17e+07,10231.0,191.6


**Interpretación:**
* **Conciliación OK:** las 4.287.395 filas, los 1.143 días y los 677 millones de casos acumulados del último día cuadran entre la fuente y el DW, y no hay hechos sin fecha ni sin territorio.
* **Chile frente a Sudamérica:** Chile es **2.º en fallecidos por 100.000** (336), detrás de Perú (666) y apenas sobre Brasil (329). En casos por 100.000 es 2.º (27.162), detrás de Uruguay, lo que también refleja cuánto testeó cada país.
* **Regiones de Chile:** Magallanes y Metropolitana encabezan los fallecidos por 100.000; **Biobío queda 10.º de 16** (309). Ojo: los nuevos por región parten el 20-may-2020, cuando Chile empieza a informar por región, así que no incluyen los fallecidos de antes de esa fecha.
* **Roll-up por año:** 2022 concentra la mayoría de los casos nuevos de Chile (ómicron), con 3,2 millones.

## 8 · Guardar

In [15]:
dw.close()
shutil.copy(DW_LOCAL, os.path.join(DW_DRIVE, 'covid_dw.db'))
print('DW copiado a Drive:', round(os.path.getsize(DW_LOCAL) / 1e6), 'MB')

DW copiado a Drive: 188 MB
